In [2]:
%pip install tensorflow

  Obtaining dependency information for tensorflow from https://files.pythonhosted.org/packages/5c/98/d145af334fd5807d6ba1ead447bf0c57a36654ea58e726d70c0d09cae913/tensorflow-2.19.0-cp312-cp312-win_amd64.whl.metadata
  Obtaining dependency information for absl-py>=1.0.0 from https://files.pythonhosted.org/packages/98/5e/34ccb5bfb8dae555045c2dd13375e01ac8e2c1f200a4e4051e95fb9addf0/absl_py-2.2.1-py3-none-any.whl.metadata
  Obtaining dependency information for astunparse>=1.6.0 from https://files.pythonhosted.org/packages/2b/03/13dde6512ad7b4557eb792fbcf0c653af6076b81e5941d36ec61f7ce6028/astunparse-1.6.3-py2.py3-none-any.whl.metadata
  Obtaining dependency information for flatbuffers>=24.3.25 from https://files.pythonhosted.org/packages/b8/25/155f9f080d5e4bc0082edfda032ea2bc2b8fab3f4d25d46c1e9dd22a1a89/flatbuffers-25.2.10-py2.py3-none-any.whl.metadata
  Obtaining dependency information for gast!=0.5.0,!=0.5.1,!=0.5.2,>=0.2.1 from https://files.pythonhosted.org/packages/a3/61/8001b38461d751c

ERROR: Could not install packages due to an OSError: [WinError 2] The system cannot find the file specified: 'c:\\Python312\\Scripts\\markdown_py.exe' -> 'c:\\Python312\\Scripts\\markdown_py.exe.deleteme'


[notice] A new release of pip is available: 23.2.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import tensorflow as tf
import numpy as np

In [2]:
# Load the TFLite model
interpreter = tf.lite.Interpreter(model_path=r"C:\Users\Asus TUF -PC\LeafSense AI training\model.tflite")
interpreter.allocate_tensors()

In [3]:
# Get input and output details
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

# Print input and output details for verification
print("Input details:", input_details)
print("Output details:", output_details)


Input details: [{'name': 'serving_default_input:0', 'index': 0, 'shape': array([  1,   3, 224, 224]), 'shape_signature': array([ -1,   3, 224, 224]), 'dtype': <class 'numpy.float32'>, 'quantization': (0.0, 0), 'quantization_parameters': {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0}, 'sparsity_parameters': {}}]
Output details: [{'name': 'PartitionedCall:0', 'index': 472, 'shape': array([ 1, 10]), 'shape_signature': array([-1, 10]), 'dtype': <class 'numpy.float32'>, 'quantization': (0.0, 0), 'quantization_parameters': {'scales': array([], dtype=float32), 'zero_points': array([], dtype=int32), 'quantized_dimension': 0}, 'sparsity_parameters': {}}]


In [4]:
# Function to preprocess the input image
def preprocess_image(image_path):
    from PIL import Image
    image = Image.open(image_path).convert("RGB")  # Ensure 3 channels (RGB)
    image = image.resize((224, 224))  # Resize to the model's input size
    image = np.array(image, dtype=np.float32) / 255.0  # Normalize to [0, 1]
    image = np.expand_dims(image, axis=0)  # Add batch dimension
    # Transpose the image to (1, 3, 224, 224)
    image = image.transpose((0, 3, 1, 2))  # Change the order of dimensions
    return image

In [7]:
# Test the model with an example image
image_path = "LeafSense/data/test/ManihotEsculenta/5.jpg"  # Replace with your test image path
input_data = preprocess_image(image_path)


In [9]:
# Ensure input shape matches the model's input shape
if input_data.shape != tuple(input_details[0]['shape']):
    print(f"Input shape mismatch: Expected {input_details[0]['shape']}, but got {input_data.shape}")
    exit()

# Set the tensor to the model's input
interpreter.set_tensor(input_details[0]['index'], input_data)

# Run inference
interpreter.invoke()

# Get the output tensor and process the results
output_data = interpreter.get_tensor(output_details[0]['index'])

# Display the results
print("Raw output:", output_data)
class_labels = ["ArtocarpusHeterophyllus", "BroussonetiaPapyrifera", "ManihotEsculenta", "SamaneaSaman"]  # Replace with your class names
predicted_class = np.argmax(output_data)
probabilities = tf.nn.softmax(output_data).numpy()
confidence = probabilities[0][predicted_class]
print(f"Predicted class: {class_labels[predicted_class]} with confidence {confidence:.2f}")



Raw output: [[ 0.08818775  0.17214662  2.561323   -2.316088    0.20891945 -1.8477956
  -0.80545384  3.7300084  -0.62203354 -1.2514032 ]]


IndexError: list index out of range